# 20 · Transformer fine-tuning (optional GPU)

**Owner:** Teammate B
**Report section:** Open-ended improvements; Model comparison

**Goal.** Fine-tune `microsoft/deberta-v3-small` (or `distilroberta-base`), max_len 256, on the shared split. Save val/test probabilities with `save_predictions` so they can be ensembled with the CPU models.

Run with *Restart & Run All*. Set `QUICK = True` for a fast debug pass on the 2,000-row
starter file, `QUICK = False` for the numbers that go in the report. Every result uses the
shared split from `src.data.get_split()` (seed 42), so numbers are comparable across notebooks.

**Running on Kaggle / Colab (GPU):**

```bash
!git clone https://github.com/2akc/CS5785_GroupProject.git && cd CS5785_GroupProject
!pip install -q -r requirements.txt -r requirements-gpu.txt
# Kaggle: attach the competition data, then
import os; os.environ["AIHUMAN_DATA_DIR"] = "/kaggle/input/<competition-slug>"
```

Afterwards, download `results/preds_deberta_small.csv` and commit it on your branch.
The final `predict.py` path stays CPU-only and consumes the saved predictions via `--ensemble`.

In [ ]:
# --- setup: make `src/` importable from anywhere (repo root, notebooks/, Kaggle/Colab clone) ---
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

QUICK = True  # True: 2,000-row starter file for fast debugging. False: full data (report numbers).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import SEED, set_seed, RESULTS_DIR, FIGURES_DIR
from src.data import get_split, load_test, load_unlabeled, load_train, get_cv_folds, xy
from src.features import build_features
from src.models import make_model
from src.evaluate import (auroc, cross_val_auroc, plot_roc, savefig, log_result,
                          save_predictions, predict_proba1)

set_seed(SEED)
SUFFIX = "_quick" if QUICK else ""   # keeps QUICK-mode outputs separate from real ones

In [ ]:
train_df, val_df = get_split(quick=QUICK)
test_df = load_test()
X_tr, y_tr = xy(train_df)
X_va, y_va = xy(val_df)

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"   # or "distilroberta-base"
MAX_LEN = 256
EPOCHS, LR, BATCH = 3, 2e-5, 16
CKPT_DIR = ROOT / "checkpoints" / MODEL_NAME.split("/")[-1]   # gitignored

try:
    import torch, transformers
    DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEVICE)
except ImportError:
    torch = None
    print("GPU requirements not installed (pip install -r requirements-gpu.txt). Skipping.")

## TODO 1: tokenize and train

* `AutoTokenizer` / `AutoModelForSequenceClassification(num_labels=2)`; build `datasets.Dataset`
  from `train_df` / `val_df`, `truncation=True, max_length=MAX_LEN`.
* `Trainer` with `output_dir=CKPT_DIR`, `save_strategy="epoch"`, `load_best_model_at_end=True`,
  `metric_for_best_model="auroc"` (compute_metrics → softmax[:, 1] → `auroc`), `fp16=True` on CUDA,
  `seed=SEED`. Resume after a Colab disconnect with `trainer.train(resume_from_checkpoint=True)`.
* Use `QUICK = True` first to make sure the loop runs end to end.

In [ ]:
# TODO

## TODO 2: predict and save

* `val_probs` from the train-split model; for `test_probs`, either refit on train+val (better) or
  reuse the same model (cheaper; note this in the report).
* `save_predictions("deberta_small" + SUFFIX, val_df, val_probs, test_df, test_probs)` and
  `log_result("transformer", config, {"val_auroc": ...})`.

In [ ]:
# TODO